# LeetCode 1315 — Sum of Nodes with Even-Valued Grandparent bằng Python

Link đề: https://leetcode.com/problems/sum-of-nodes-with-even-valued-grandparent/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Breadth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def sumEvenGrandparent(self, root: TreeNode | None) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu quan hệ **cha – ông – cháu** trong cây nhị phân và cách "nhìn" tổ tiên khi duyệt từ trên xuống.
2. Kỹ thuật **DFS mang theo trạng thái** (giá trị cha, giá trị ông) — dịch chuyển trạng thái `(ông, cha) → (cha, nút)` khi đi xuống.
3. Viết cùng ý tưởng bằng **BFS / stack** để tránh giới hạn đệ quy với cây lệch `10^4` nút.
4. Đổi góc nhìn: **từ ông nhìn xuống cháu** (mỗi nút chẵn cộng tối đa 4 cháu).
5. Tránh lỗi: chọn **giá trị lính canh** sai (0 là số chẵn!), đảo thứ tự tham số, kiểm tra cha thay vì ông, truy cập `None.left`.

## 2. Đề bài & ví dụ

Cho gốc `root` của cây nhị phân, trả về **tổng giá trị các nút có ông (grandparent) mang giá trị chẵn**. Nếu không có nút nào như vậy, trả về `0`. **Ông** của một nút là cha của cha nó (nếu tồn tại).

**Ví dụ 1:** `root = [6,7,8,2,7,1,3,9,null,1,4,null,null,null,5]` → `18`

```
              6
          /       \
         7         8
       /   \      /  \
      2     7    1    3
     /     / \         \
    9     1   4         5
```
Ông chẵn: `6` (cháu `2, 7, 1, 3`), `8` (cháu `5`), `2` (không có cháu). Tổng `2 + 7 + 1 + 3 + 5 = 18`.

**Ví dụ 2:** `root = [1]` → `0`.

**Ràng buộc:** số nút trong `[1, 10^4]`, `1 <= Node.val <= 100`.

In [1]:
import random
import sys
import time
from collections import deque
from typing import List, Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = trống)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            q.append(node.right)
        i += 1
    return root


def show(root, indent=0):
    if root is None:
        return
    show(root.right, indent + 4)
    print(" " * indent + str(root.val))
    show(root.left, indent + 4)


EX1 = [6, 7, 8, 2, 7, 1, 3, 9, None, 1, 4, None, None, None, 5]
examples = [(EX1, 18), ([1], 0)]
show(build_tree(EX1))

            5
        3
    8
        1
6
            4
        7
            1
    7
        2
            9


## 3. Kiến thức nền

### 3.1 Cha, ông, cháu

Với nút `x`: con của `x` là `x.left`, `x.right`; **cháu** của `x` là con của con: `x.left.left`, `x.left.right`, `x.right.left`, `x.right.right` (tối đa 4, có thể là `None`). Ngược lại, `x` là **ông** của các cháu đó. Chú ý: phải kiểm tra `x.left is not None` trước khi truy cập `x.left.left`.

In [2]:
def grandchildren(x):
    res = []
    for child in (x.left, x.right):
        if child:
            for g in (child.left, child.right):
                if g:
                    res.append(g.val)
    return res

root = build_tree(EX1)
print("cháu của 6:", grandchildren(root))
print("cháu của 8:", grandchildren(root.right))
print("cháu của 2:", grandchildren(root.left.left))

cháu của 6: [2, 7, 1, 3]
cháu của 8: [5]
cháu của 2: []


### 3.2 Kiểm tra chẵn lẻ

`v % 2 == 0` ⇔ `v` chẵn. Cũng có thể dùng bit cuối: `v & 1 == 0`. **Chú ý:** `0` là số **chẵn** — điều này quan trọng khi chọn giá trị "lính canh" cho tổ tiên không tồn tại.

In [3]:
for v in [6, 7, 0, 100]:
    print(v, "chẵn?" , v % 2 == 0, "| bit cuối:", v & 1)

6 chẵn? True | bit cuối: 0
7 chẵn? False | bit cuối: 1
0 chẵn? True | bit cuối: 0
100 chẵn? True | bit cuối: 0


### 3.3 DFS đệ quy truyền tham số xuống

Thông tin "từ trên xuống" (độ sâu, tổ tiên, đường đi…) được truyền qua **tham số** của hàm đệ quy. Mỗi lần đi xuống, ta tính tham số mới cho con. Kết quả "từ dưới lên" (tổng) được **trả về** và cộng dồn.

In [4]:
def sum_with_depth(node, depth):
    # ví dụ: tổng (giá trị * độ sâu) — độ sâu truyền xuống, tổng trả lên
    if node is None:
        return 0
    return node.val * depth + sum_with_depth(node.left, depth + 1) + sum_with_depth(node.right, depth + 1)

print(sum_with_depth(build_tree([1, 2, 3]), 0), "= 1*0 + 2*1 + 3*1")

5 = 1*0 + 2*1 + 3*1


### 3.4 BFS/stack với tuple trạng thái

Thay vì tham số đệ quy, ta đẩy **tuple** `(node, parent_val, grand_val)` vào hàng đợi/stack. Không đệ quy ⇒ không lo `RecursionError`.

In [5]:
q = deque([(build_tree([1, 2, 3]), None, None)])
while q:
    node, p, g = q.popleft()
    print(f"nút {node.val}: cha={p}, ông={g}")
    for c in (node.left, node.right):
        if c:
            q.append((c, node.val, p))

nút 1: cha=None, ông=None
nút 2: cha=1, ông=None
nút 3: cha=1, ông=None


### 3.5 Giá trị lính canh (sentinel)

Gốc không có cha/ông. Ta cần một giá trị đại diện cho "không tồn tại" sao cho **điều kiện "ông chẵn" luôn sai**. Lựa chọn an toàn: `None` (rồi kiểm tra `g is not None`) hoặc một số **lẻ** như `1` / `-1`. **Không** dùng `0`.

In [6]:
for sentinel in (None, 1, 0):
    is_even = sentinel is not None and sentinel % 2 == 0
    print(f"lính canh {sentinel!r}: bị coi là ông chẵn? {is_even}")

lính canh None: bị coi là ông chẵn? False
lính canh 1: bị coi là ông chẵn? False
lính canh 0: bị coi là ông chẵn? True


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** — lập bảng mỗi nút với cha và ông:

| Nút | Cha | Ông | Ông chẵn? | Cộng |
|---|---|---|---|---|
| 6 | – | – | ✘ | |
| 7 (trái) | 6 | – | ✘ | |
| 8 | 6 | – | ✘ | |
| 2 | 7 | 6 | ✔ | 2 |
| 7 (phải của 7) | 7 | 6 | ✔ | 7 |
| 1 (trái của 8) | 8 | 6 | ✔ | 1 |
| 3 | 8 | 6 | ✔ | 3 |
| 9 | 2 | 7 | ✘ | |
| 1 (trái của 7) | 7 | 7 | ✘ | |
| 4 | 7 | 7 | ✘ | |
| 5 | 3 | 8 | ✔ | 5 |

Tổng `2 + 7 + 1 + 3 + 5 = 18` ✔. Nhận xét: khi đi từ một nút xuống con, **ông của con = cha của nút**, **cha của con = nút** ⇒ chỉ cần mang theo 2 giá trị.

In [7]:
rows = []
q = deque([(build_tree(EX1), None, None)])
while q:
    node, p, g = q.popleft()
    ok = g is not None and g % 2 == 0
    rows.append((node.val, p, g, ok))
    for c in (node.left, node.right):
        if c:
            q.append((c, node.val, p))
for r in rows:
    print(f"nút {r[0]:>2} cha={str(r[1]):>4} ông={str(r[2]):>4} -> {'CỘNG' if r[3] else ''}")
print("tổng =", sum(r[0] for r in rows if r[3]))

nút  6 cha=None ông=None -> 
nút  7 cha=   6 ông=None -> 
nút  8 cha=   6 ông=None -> 
nút  2 cha=   7 ông=   6 -> CỘNG
nút  7 cha=   7 ông=   6 -> CỘNG
nút  1 cha=   8 ông=   6 -> CỘNG
nút  3 cha=   8 ông=   6 -> CỘNG
nút  9 cha=   2 ông=   7 -> 
nút  1 cha=   7 ông=   7 -> 
nút  4 cha=   7 ông=   7 -> 
nút  5 cha=   3 ông=   8 -> CỘNG
tổng = 18


## 5. Cách 1 — Brute Force / baseline: tìm cha bằng cách duyệt lại cả cây

**Ý tưởng:** với mỗi nút `x`, viết hàm `find_parent(root, x)` duyệt toàn bộ cây để tìm nút có con là `x`. Ông = `find_parent(root, find_parent(root, x))`. Nếu tồn tại và chẵn thì cộng `x.val`.

Cách này "nghĩ thẳng theo định nghĩa", không cần ý tưởng gì, nhưng mỗi lần tìm cha tốn `O(n)`.

In [8]:
def all_nodes(root):
    out, stack = [], [root] if root else []
    while stack:
        node = stack.pop()
        out.append(node)
        if node.right: stack.append(node.right)
        if node.left: stack.append(node.left)
    return out

def find_parent(root, x):
    for node in all_nodes(root):
        if node.left is x or node.right is x:
            return node
    return None

def sum_even_gp_brute(root):
    total = 0
    for x in all_nodes(root):
        p = find_parent(root, x)
        g = find_parent(root, p) if p else None
        if g is not None and g.val % 2 == 0:
            total += x.val
    return total

for vals, want in examples:
    print(sum_even_gp_brute(build_tree(vals)), "| mong đợi", want)

18 | mong đợi 18
0 | mong đợi 0


**Độ phức tạp:** Time `O(n²)` (mỗi nút gọi `find_parent` hai lần, mỗi lần `O(n)`) ⇒ `10^8` với `n = 10^4`: quá chậm. Space `O(n)`.

**Nút thắt:** ta "quên" thông tin cha/ông mỗi khi rời khỏi nút. Nếu **mang theo** thông tin đó trong lúc duyệt từ trên xuống thì không cần tìm lại.

In [9]:
def random_tree(rng, n):
    root = TreeNode(rng.randint(1, 100))
    slots = [(root, "L"), (root, "R")]
    for _ in range(n - 1):
        parent, side = slots.pop(rng.randrange(len(slots)))
        child = TreeNode(rng.randint(1, 100))
        setattr(parent, "left" if side == "L" else "right", child)
        slots += [(child, "L"), (child, "R")]
    return root

rng = random.Random(1)
t500 = random_tree(rng, 500)
t0 = time.perf_counter()
sum_even_gp_brute(t500)
print(f"brute n=500: {time.perf_counter() - t0:.3f}s (n=10^4 sẽ chậm gấp ~400 lần)")

brute n=500: 0.046s (n=10^4 sẽ chậm gấp ~400 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (mang trạng thái xuống):** khi DFS đi từ nút `x` xuống con `c`:
- `cha(c) = x`
- `ông(c) = cha(x)`

⇒ Hàm `dfs(node, parent_val, grand_val)`; gọi con bằng `dfs(child, node.val, parent_val)`. Đây là phép **dịch trạng thái** `(ông, cha) → (cha, nút)`.

**Quy tắc:** *nếu* `grand_val` tồn tại và chẵn *thì* cộng `node.val`.

**Insight 2 (đổi góc nhìn):** thay vì mỗi nút hỏi "ông tôi có chẵn không?", để **mỗi nút chẵn tự cộng các cháu của nó** (tối đa 4). Mỗi cháu có đúng một ông ⇒ không bị cộng trùng.

**Invariant DFS:** khi `dfs(node, p, g)` được gọi, `p` đúng là giá trị cha của `node` và `g` đúng là giá trị ông (hoặc lính canh nếu không tồn tại).

In [10]:
# Minh hoạ dịch trạng thái trên nhánh trái-phải của ví dụ 1: 6 -> 7 -> 7 -> 4
path = [6, 7, 7, 4]
g, p = None, None
for v in path:
    print(f"tại nút {v}: (ông, cha) = ({g}, {p})")
    g, p = p, v

tại nút 6: (ông, cha) = (None, None)
tại nút 7: (ông, cha) = (None, 6)
tại nút 7: (ông, cha) = (6, 7)
tại nút 4: (ông, cha) = (7, 7)


## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS đệ quy mang (cha, ông)

Dùng lính canh `1` (số lẻ) cho tổ tiên không tồn tại — giá trị nút trong đề `>= 1` nhưng lính canh chỉ dùng để kiểm tra chẵn lẻ nên an toàn.

In [11]:
def sum_even_gp_dfs(root):
    def dfs(node, parent_val, grand_val):
        if node is None:
            return 0
        s = node.val if grand_val % 2 == 0 else 0
        return s + dfs(node.left, node.val, parent_val) + dfs(node.right, node.val, parent_val)
    return dfs(root, 1, 1)                      # 1 = lính canh lẻ

for vals, want in examples:
    print(sum_even_gp_dfs(build_tree(vals)), "| mong đợi", want)

18 | mong đợi 18
0 | mong đợi 0


Code nộp LeetCode (cách 2, nâng giới hạn đệ quy cho cây lệch):

In [12]:
class SolutionDFS:
    def sumEvenGrandparent(self, root: Optional[TreeNode]) -> int:
        sys.setrecursionlimit(max(sys.getrecursionlimit(), 30000))
        def dfs(node, parent_val, grand_val):
            if node is None:
                return 0
            s = node.val if grand_val % 2 == 0 else 0
            return s + dfs(node.left, node.val, parent_val) + dfs(node.right, node.val, parent_val)
        return dfs(root, 1, 1)

print(SolutionDFS().sumEvenGrandparent(build_tree(EX1)))

18


**Độ phức tạp cách 2:** Time `O(n)`, Space `O(h)` cho stack đệ quy (`h` = chiều cao, cây lệch `O(n)`).

### 7.2 Cách 3 — BFS (hoặc stack) với tuple `(node, cha, ông)`

Cùng ý tưởng nhưng lặp, an toàn với cây sâu `10^4`.

In [13]:
def sum_even_gp_bfs(root):
    total = 0
    q = deque([(root, None, None)])
    while q:
        node, p, g = q.popleft()
        if g is not None and g % 2 == 0:
            total += node.val
        for c in (node.left, node.right):
            if c:
                q.append((c, node.val, p))      # dịch trạng thái
    return total

for vals, want in examples:
    print(sum_even_gp_bfs(build_tree(vals)), "| mong đợi", want)

18 | mong đợi 18
0 | mong đợi 0


Code nộp LeetCode (cách 3):

In [14]:
class SolutionBFS:
    def sumEvenGrandparent(self, root: Optional[TreeNode]) -> int:
        total = 0
        q = deque([(root, None, None)])
        while q:
            node, p, g = q.popleft()
            if g is not None and g % 2 == 0:
                total += node.val
            if node.left:
                q.append((node.left, node.val, p))
            if node.right:
                q.append((node.right, node.val, p))
        return total

print(SolutionBFS().sumEvenGrandparent(build_tree(EX1)))

18


**Độ phức tạp cách 3:** Time `O(n)`, Space `O(w)` (độ rộng lớn nhất của cây).

### 7.3 Cách 4 — Từ ông nhìn xuống cháu

Duyệt mọi nút (bất kỳ thứ tự). Với mỗi nút **chẵn**, cộng giá trị các cháu (kiểm tra `None` cẩn thận). Không cần mang trạng thái.

In [15]:
def sum_even_gp_down(root):
    total = 0
    stack = [root]
    while stack:
        node = stack.pop()
        if node.val % 2 == 0:
            for child in (node.left, node.right):
                if child:
                    if child.left:
                        total += child.left.val
                    if child.right:
                        total += child.right.val
        if node.left: stack.append(node.left)
        if node.right: stack.append(node.right)
    return total

for vals, want in examples:
    print(sum_even_gp_down(build_tree(vals)), "| mong đợi", want)

18 | mong đợi 18
0 | mong đợi 0


Code nộp LeetCode (cách 4):

In [16]:
class SolutionDown:
    def sumEvenGrandparent(self, root: Optional[TreeNode]) -> int:
        total = 0
        stack = [root]
        while stack:
            node = stack.pop()
            if node.val % 2 == 0:
                for child in (node.left, node.right):
                    if child:
                        for g in (child.left, child.right):
                            if g:
                                total += g.val
            for child in (node.left, node.right):
                if child:
                    stack.append(child)
        return total

print(SolutionDown().sumEvenGrandparent(build_tree(EX1)))

18


**Độ phức tạp cách 4:** Time `O(n)` (mỗi nút làm tối đa 6 phép kiểm tra con/cháu), Space `O(h)` cho stack.

In [17]:
rng = random.Random(2)
big = random_tree(rng, 10**4)
for name, f in [("dfs", sum_even_gp_dfs), ("bfs", sum_even_gp_bfs), ("down", sum_even_gp_down)]:
    t0 = time.perf_counter()
    r = f(big)
    print(f"{name:5s}: {r} ({time.perf_counter() - t0:.4f}s)")

dfs  : 255078 (0.0015s)
bfs  : 255078 (0.0018s)
down : 255078 (0.0018s)


## 8. TRACE từng bước

Trace DFS (cách 2) trên Ví dụ 1. Mỗi dòng: nút đang thăm, `(cha, ông)` nhận được, có cộng không, và tổng cộng dồn theo thứ tự thăm (preorder).

| Thứ tự | Nút | (cha, ông) | Ông chẵn? | Tổng dồn |
|---|---|---|---|---|
| 1 | 6 | (–, –) | ✘ | 0 |
| 2 | 7 | (6, –) | ✘ | 0 |
| 3 | 2 | (7, 6) | ✔ | 2 |
| 4 | 9 | (2, 7) | ✘ | 2 |
| 5 | 7 | (7, 6) | ✔ | 9 |
| 6 | 1 | (7, 7) | ✘ | 9 |
| 7 | 4 | (7, 7) | ✘ | 9 |
| 8 | 8 | (6, –) | ✘ | 9 |
| 9 | 1 | (8, 6) | ✔ | 10 |
| 10 | 3 | (8, 6) | ✔ | 13 |
| 11 | 5 | (3, 8) | ✔ | **18** |

In [18]:
def trace_dfs(root):
    state = {"total": 0, "step": 0}
    def dfs(node, p, g, depth):
        if node is None:
            return
        state["step"] += 1
        ok = g is not None and g % 2 == 0
        if ok:
            state["total"] += node.val
        print(f"{state['step']:>2}. {'  ' * depth}nút {node.val} (cha={p}, ông={g}) "
              f"{'-> CỘNG ' + str(node.val) if ok else ''} | tổng={state['total']}")
        dfs(node.left, node.val, p, depth + 1)
        dfs(node.right, node.val, p, depth + 1)
    dfs(root, None, None, 0)
    print("=> đáp án:", state["total"])
    return state["total"]

trace_dfs(build_tree(EX1))

 1. nút 6 (cha=None, ông=None)  | tổng=0
 2.   nút 7 (cha=6, ông=None)  | tổng=0
 3.     nút 2 (cha=7, ông=6) -> CỘNG 2 | tổng=2
 4.       nút 9 (cha=2, ông=7)  | tổng=2
 5.     nút 7 (cha=7, ông=6) -> CỘNG 7 | tổng=9
 6.       nút 1 (cha=7, ông=7)  | tổng=9
 7.       nút 4 (cha=7, ông=7)  | tổng=9
 8.   nút 8 (cha=6, ông=None)  | tổng=9
 9.     nút 1 (cha=8, ông=6) -> CỘNG 1 | tổng=10
10.     nút 3 (cha=8, ông=6) -> CỘNG 3 | tổng=13
11.       nút 5 (cha=3, ông=8) -> CỘNG 5 | tổng=18
=> đáp án: 18


18

Trace cách 4 (từ ông nhìn xuống):

In [19]:
def trace_down(root):
    total = 0
    stack = [root]
    while stack:
        node = stack.pop()
        if node.val % 2 == 0:
            gs = [g.val for c in (node.left, node.right) if c for g in (c.left, c.right) if g]
            total += sum(gs)
            print(f"nút chẵn {node.val}: cháu = {gs} -> tổng = {total}")
        for c in (node.right, node.left):
            if c:
                stack.append(c)
    print("=> đáp án:", total)
    return total

trace_down(build_tree(EX1))

nút chẵn 6: cháu = [2, 7, 1, 3] -> tổng = 13
nút chẵn 2: cháu = [] -> tổng = 13
nút chẵn 4: cháu = [] -> tổng = 13
nút chẵn 8: cháu = [5] -> tổng = 18
=> đáp án: 18


18

## 9. Code nộp LeetCode

Bản chính thức: BFS mang `(node, cha, ông)` — không đệ quy, lính canh `None` rõ ràng.

In [20]:
class Solution:
    def sumEvenGrandparent(self, root: TreeNode | None) -> int:
        total = 0
        q = deque([(root, None, None)])          # (nút, giá trị cha, giá trị ông)
        while q:
            node, parent, grand = q.popleft()
            if grand is not None and grand % 2 == 0:
                total += node.val
            # đi xuống: ông mới = cha cũ, cha mới = nút hiện tại
            if node.left:
                q.append((node.left, node.val, parent))
            if node.right:
                q.append((node.right, node.val, parent))
        return total

sol = Solution()
for vals, want in examples:
    got = sol.sumEvenGrandparent(build_tree(vals))
    print(got, "✔" if got == want else "✘")

18 ✔
0 ✔


## 10. Vì sao thuật toán đúng

**Cách 2/3 (mang trạng thái):** quy nạp theo độ sâu.
- Gốc nhận `(None, None)` — đúng vì gốc không có cha, ông.
- Nếu `node` nhận đúng `(cha(node), ông(node))` thì con `c` nhận `(node.val, cha(node))` = `(cha(c), ông(c))` theo định nghĩa ông = cha của cha.

Vậy mọi nút nhận đúng giá trị ông; thuật toán thăm mỗi nút đúng một lần và cộng `node.val` khi và chỉ khi ông tồn tại và chẵn ⇒ đúng tổng cần tìm.

**Cách 4 (nhìn xuống):** ánh xạ "cháu → ông" là hàm (mỗi nút có tối đa một ông). Tổng theo các cặp (ông chẵn, cháu) gom theo ông hay theo cháu đều như nhau, và mỗi cháu chỉ xuất hiện trong danh sách cháu của đúng một ông ⇒ không trùng, không sót.

## 11. Phân tích Time / Space

Gọi `n` = số nút (≤ `10^4`), `h` = chiều cao, `w` = độ rộng lớn nhất.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Tìm cha bằng duyệt lại | `O(n²)` | `O(n)` | baseline |
| 2. DFS đệ quy (cha, ông) | `O(n)` | `O(h)` | cần nâng recursion limit |
| 3. BFS tuple (cha, ông) | `O(n)` | `O(w)` | bản nộp chính |
| 4. Từ ông nhìn xuống | `O(n)` | `O(h)` | không cần trạng thái |

Cây cân bằng: `h ≈ log n`, `w ≈ n/2`; cây lệch: `h ≈ n`, `w = 1`.

## 12. Bộ test

Ví dụ đề, edge case (1 nút, 2 tầng — chưa có cháu, cây lệch, mọi giá trị chẵn, mọi giá trị lẻ, ông chẵn nhưng cha lẻ và ngược lại), cây lệch `10^4` nút, và đối chiếu với brute force trên cây ngẫu nhiên.

In [21]:
ALL = [sum_even_gp_brute, sum_even_gp_dfs, sum_even_gp_bfs, sum_even_gp_down,
       Solution().sumEvenGrandparent, SolutionDFS().sumEvenGrandparent,
       SolutionBFS().sumEvenGrandparent, SolutionDown().sumEvenGrandparent]
tests = [
    (EX1, 18),
    ([1], 0),
    ([2, 4, 6], 0),                      # chưa có cháu
    ([2, 1, 3, 4, 5], 9),                # ông 2 chẵn, cha 1 lẻ
    ([1, 2, 3, 4, 5], 0),                # cha chẵn nhưng ông lẻ
    ([2, 2, 2, 2, 2, 2, 2], 8),          # mọi nút chẵn: 4 cháu
    ([3, 5, 7, 9, 11], 0),               # mọi nút lẻ
    ([2, None, 4, None, 6, None, 8], 14),# lệch phải: 6 (ông 2) + 8 (ông 4)
    ([4, 1, None, 2, None, 3], 2 + 0),   # 2 có ông 4; 3 có ông 1
]
for vals, want in tests:
    for f in ALL:
        got = f(build_tree(vals))
        assert got == want, (f, vals, got, want)
print("Tất cả", len(tests), "test cố định qua cho", len(ALL), "hàm")

Tất cả 9 test cố định qua cho 8 hàm


In [22]:
rng = random.Random(1315)
for _ in range(400):
    root = random_tree(rng, rng.randint(1, 50))
    want = sum_even_gp_brute(root)
    for f in ALL[1:]:
        assert f(root) == want
print("Đối chiếu 400 cây ngẫu nhiên: khớp")

# cây lệch trái 10^4 nút với giá trị 2 (mọi nút từ tầng 2 trở đi đều có ông chẵn)
chain = TreeNode(2)
cur = chain
for _ in range(10**4 - 1):
    cur.left = TreeNode(2)
    cur = cur.left
t0 = time.perf_counter()
r = [Solution().sumEvenGrandparent(chain), SolutionDFS().sumEvenGrandparent(chain), SolutionDown().sumEvenGrandparent(chain)]
assert r == [2 * (10**4 - 2)] * 3
print("Cây lệch 10^4 nút:", r[0], f"({time.perf_counter() - t0:.3f}s)")

Đối chiếu 400 cây ngẫu nhiên: khớp


Cây lệch 10^4 nút: 19996 (0.012s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `0` làm lính canh cho tổ tiên không tồn tại

In [23]:
def wrong_sentinel(root):
    def dfs(node, p, g):
        if node is None:
            return 0
        s = node.val if g % 2 == 0 else 0
        return s + dfs(node.left, node.val, p) + dfs(node.right, node.val, p)
    return dfs(root, 0, 0)                 # SAI: 0 là số chẵn!

for vals in ([1], [1, 3, 5], EX1):
    r = build_tree(vals)
    print(vals[:5], "-> sai:", wrong_sentinel(r), "| đúng:", Solution().sumEvenGrandparent(r))

[1] -> sai: 1 | đúng: 0
[1, 3, 5] -> sai: 9 | đúng: 0
[6, 7, 8, 2, 7] -> sai: 39 | đúng: 18


**Vì sao sai:** gốc và các con của gốc nhận "ông = 0", mà `0 % 2 == 0` ⇒ bị cộng oan. Với `[1]` ra 1 thay vì 0; với `[1,3,5]` ra 9 thay vì 0. **Sửa:** dùng `None` (và kiểm tra `is not None`) hoặc một số **lẻ** như `1`/`-1`.

### Lỗi 2 — Đảo thứ tự tham số khi đi xuống

In [24]:
def wrong_shift(root):
    def dfs(node, p, g):
        if node is None:
            return 0
        s = node.val if g is not None and g % 2 == 0 else 0
        return s + dfs(node.left, p, node.val) + dfs(node.right, p, node.val)   # SAI: đảo (cha, ông)
    return dfs(root, None, None)

r = build_tree(EX1)
print("sai :", wrong_shift(r))
print("đúng:", Solution().sumEvenGrandparent(r))

sai : 28
đúng: 18


**Vì sao sai:** đi xuống con thì **cha mới = nút hiện tại**, **ông mới = cha cũ**: `dfs(child, node.val, p)`. Viết ngược thành `dfs(child, p, node.val)` thì "ông" của con thực ra là cha của nó ⇒ ra tổng các nút có **cha** chẵn (sai lệch tuỳ cây). **Sửa:** nhớ câu "**ông ← cha, cha ← tôi**".

### Lỗi 3 — Kiểm tra cha chẵn thay vì ông chẵn

In [25]:
def wrong_parent(root):
    total = 0
    q = deque([(root, None, None)])
    while q:
        node, p, g = q.popleft()
        if p is not None and p % 2 == 0:       # SAI: đề hỏi ông, không phải cha
            total += node.val
        for c in (node.left, node.right):
            if c:
                q.append((c, node.val, p))
    return total

r = build_tree(EX1)
print("sai :", wrong_parent(r), "| đúng:", Solution().sumEvenGrandparent(r))
r = build_tree([1, 2, 3, 4, 5])
print("sai :", wrong_parent(r), "| đúng:", Solution().sumEvenGrandparent(r))

sai : 28 | đúng: 18
sai : 9 | đúng: 0


**Vì sao sai:** đọc lướt đề. Với `[1,2,3,4,5]`, cha của `4, 5` là `2` (chẵn) nhưng ông là `1` (lẻ) ⇒ đáp án đúng là 0. **Sửa:** điều kiện trên `grand`.

### Lỗi 4 — Cách "nhìn xuống" truy cập `None.left`

In [26]:
def wrong_none(root):
    total = 0
    stack = [root]
    while stack:
        node = stack.pop()
        if node.val % 2 == 0:
            total += (node.left.left.val if node.left.left else 0)     # SAI: node.left có thể là None
            total += (node.left.right.val if node.left.right else 0)
            total += (node.right.left.val if node.right.left else 0)
            total += (node.right.right.val if node.right.right else 0)
        for c in (node.left, node.right):
            if c:
                stack.append(c)
    return total

try:
    print(wrong_none(build_tree(EX1)))
except AttributeError as e:
    print("Bắt được AttributeError:", e)
print("đúng:", Solution().sumEvenGrandparent(build_tree(EX1)))

Bắt được AttributeError: 'NoneType' object has no attribute 'left'
đúng: 18


**Vì sao sai:** nút chẵn `2` (trong ví dụ) có `right = None`, nên `node.right.left` ném lỗi. **Sửa:** kiểm tra `child` khác `None` trước khi xét cháu (như `SolutionDown`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Điều kiện phụ thuộc **tổ tiên** (cha, ông, đường đi từ gốc) ⇒ **truyền tham số xuống** trong DFS/BFS.
- Điều kiện phụ thuộc **con cháu** ⇒ có thể **đổi góc nhìn**: để tổ tiên tự xử lý con cháu, hoặc trả kết quả từ dưới lên.
- Câu thần chú dịch trạng thái: **"ông ← cha, cha ← tôi"**.
- Lính canh: chọn giá trị **không thể thoả điều kiện** (`None`, số lẻ) — cẩn thận `0`.
- Cây tới `10^4` nút trong Python ⇒ ưu tiên BFS/stack.

## 15. Biến thể của bài

1. **Tổng nút có cha chẵn** — chỉ cần mang 1 giá trị.
2. **Tổng nút có tổ tiên đời thứ `k` chẵn** — mang theo **đường đi** từ gốc (bài tập 5).
3. **Tổng nút có ít nhất một tổ tiên chẵn** — mang cờ boolean "đã gặp số chẵn trên đường".
4. **LeetCode 1448 — Count Good Nodes in Binary Tree**: mang theo `max` trên đường đi.
5. **LeetCode 1026 — Maximum Difference Between Node and Ancestor**: mang theo `(min, max)` của tổ tiên.
6. **LeetCode 863 — All Nodes Distance K**: tổng quát "khoảng cách k" theo mọi hướng.

In [27]:
# Biến thể 3: tổng các nút có ÍT NHẤT một tổ tiên chẵn (mang cờ)
def sum_any_even_ancestor(root):
    total = 0
    stack = [(root, False)]
    while stack:
        node, seen_even = stack.pop()
        if seen_even:
            total += node.val
        flag = seen_even or node.val % 2 == 0
        for c in (node.left, node.right):
            if c:
                stack.append((c, flag))
    return total

print(sum_any_even_ancestor(build_tree(EX1)), "= tổng mọi nút trừ gốc 6 (vì 6 chẵn) =", 7+8+2+7+1+3+9+1+4+5)

47 = tổng mọi nút trừ gốc 6 (vì 6 chẵn) = 47


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tìm cha bằng duyệt lại | Theo định nghĩa | `O(n²)` | `O(n)` | Kiểm chứng |
| 2. DFS (cha, ông) | Dịch trạng thái khi đi xuống | `O(n)` | `O(h)` | Code ngắn, cây không quá sâu |
| 3. BFS tuple | Như 2 nhưng lặp | `O(n)` | `O(w)` | Bản nộp chuẩn |
| 4. Ông nhìn xuống cháu | Nút chẵn cộng ≤ 4 cháu | `O(n)` | `O(h)` | Không muốn mang trạng thái |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().sumEvenGrandparent(build_tree([2, 4, 6]))` và `build_tree([2, 1, 3, 4, 5])`?

**Bài 2 (tính tay):** Với cây `[8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13]`, liệt kê các cặp (ông chẵn → cháu) và tính tổng.

**Bài 3 (điền TODO):** Hoàn thành `grandchildren_vals(node)` trả list giá trị các cháu (tối đa 4) của `node`.

**Bài 4 (tự viết lại):** Viết lại DFS mang (cha, ông) bằng **stack tường minh**: `my_sum_even_gp(root)`.

**Bài 5 (mini challenge):** Viết `sum_even_kth_ancestor(root, k)`: tổng các nút có **tổ tiên đời thứ `k`** (k = 1: cha, k = 2: ông, …) mang giá trị chẵn.

In [28]:
# Bài 3, 4, 5 — khung để tự điền (KHÔNG gọi khi chạy notebook)
def grandchildren_vals_todo(node):
    # TODO: duyệt con khác None, rồi cháu khác None
    pass

def my_sum_even_gp_todo(root):
    # TODO: stack [(root, None, None)], dịch trạng thái "ông <- cha, cha <- tôi"
    pass

def sum_even_kth_ancestor_todo(root, k):
    # TODO: mang theo đường đi (list giá trị tổ tiên); tổ tiên đời k là path[-k]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) `[2,4,6]` chỉ có 2 tầng. (2) Trong `[2,1,3,4,5]`, cháu của `2` là `4, 5`. (3) Đáp án 0 và 9.

**Bài 2:** (1) Ông chẵn: `8`, `10`, `6`, `14`, `4`. (2) Cháu của `8`: `1, 6, 14`; của `10`: `13`; của `6`, `14`, `4`: không có. (3) Tổng `1 + 6 + 14 + 13 = 34`.

**Bài 3:** (1) `for child in (node.left, node.right)`. (2) `if child:` rồi xét `child.left`, `child.right`. (3) Chỉ thêm cháu khác `None`.

**Bài 4:** (1) Stack chứa tuple 3 phần tử. (2) Kiểm tra `grand` trước khi đẩy con. (3) Đẩy `(child, node.val, parent)`.

**Bài 5:** (1) DFS với `path` = danh sách giá trị từ gốc tới cha. (2) Nếu `len(path) >= k` và `path[-k]` chẵn thì cộng. (3) Bản lặp: lưu `path` dạng tuple trong stack, hoặc giữ một list và `append`/`pop` khi vào/ra (đệ quy).

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
assert Solution().sumEvenGrandparent(build_tree([2, 4, 6])) == 0
assert Solution().sumEvenGrandparent(build_tree([2, 1, 3, 4, 5])) == 9
# Bài 2
T2 = build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13])
show(T2)
assert Solution().sumEvenGrandparent(T2) == 34

# Bài 3
def grandchildren_vals(node):
    res = []
    for child in (node.left, node.right):
        if child:
            for g in (child.left, child.right):
                if g:
                    res.append(g.val)
    return res

assert grandchildren_vals(build_tree(EX1)) == [2, 7, 1, 3]
assert grandchildren_vals(build_tree([1])) == []
print("Bài 1–3: OK")

        14
            13
    10
8
            7
        6
            4
    3
        1
Bài 1–3: OK


In [30]:
# Bài 4
def my_sum_even_gp(root):
    total = 0
    stack = [(root, None, None)]
    while stack:
        node, parent, grand = stack.pop()
        if grand is not None and grand % 2 == 0:
            total += node.val
        for c in (node.left, node.right):
            if c:
                stack.append((c, node.val, parent))
    return total

# Bài 5
def sum_even_kth_ancestor(root, k):
    total = 0
    path = []
    def dfs(node):
        nonlocal total
        if node is None:
            return
        if len(path) >= k and path[-k] % 2 == 0:
            total += node.val
        path.append(node.val)
        dfs(node.left)
        dfs(node.right)
        path.pop()
    dfs(root)
    return total

def kth_brute(root, k):
    nodes = all_nodes(root)
    total = 0
    for x in nodes:
        a = x
        for _ in range(k):
            a = find_parent(root, a) if a else None
        if a is not None and a.val % 2 == 0:
            total += x.val
    return total

rng = random.Random(7)
for _ in range(200):
    root = random_tree(rng, rng.randint(1, 30))
    assert my_sum_even_gp(root) == sum_even_gp_brute(root)
    assert sum_even_kth_ancestor(root, 2) == sum_even_gp_brute(root)
    k = rng.randint(1, 4)
    assert sum_even_kth_ancestor(root, k) == kth_brute(root, k)
print("Bài 4–5: OK — mọi assert đều qua")

Bài 4–5: OK — mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "mang tổ tiên xuống":**

```python
q = deque([(root, None, None)])        # (nút, cha, ông)
while q:
    node, parent, grand = q.popleft()
    if grand is not None and COND(grand):
        total += node.val
    for c in (node.left, node.right):
        if c:
            q.append((c, node.val, parent))   # ông <- cha, cha <- tôi
```

**Template "ông nhìn xuống cháu":**

```python
for child in (node.left, node.right):
    if child:
        for g in (child.left, child.right):
            if g: total += g.val
```

**Checklist trước khi submit:**
- [ ] Lính canh cho tổ tiên không tồn tại **không** thoả điều kiện (không dùng `0`).
- [ ] Dịch trạng thái đúng thứ tự: `(child, node.val, parent)`.
- [ ] Điều kiện trên **ông**, không phải cha.
- [ ] Kiểm tra `None` trước khi truy cập con/cháu.
- [ ] Cây sâu `10^4`: dùng BFS/stack hoặc nâng recursion limit.
- [ ] Đúng chữ ký: `sumEvenGrandparent(self, root: TreeNode | None) -> int`.